# Smoke test: `opentelemetry_instrumentation_httpx`

Exercises the public API of the OpenTelemetry HTTPX instrumentation: the `HTTPXClientInstrumentor`, the sync/async client wrapping helpers, and request/response hook integration.

In [ ]:
# Setup: retry helpers for the network examples below. Public test endpoints are
# occasionally slow or flaky, so transient network errors and 429/5xx answers are retried.
import asyncio
import time

_RETRY_STATUS = (429, 500, 502, 503, 504)


def _retry(fn, tries=5, delay=1.0):
    for i in range(tries):
        try:
            resp = fn()
            if getattr(resp, 'status_code', 200) in _RETRY_STATUS and i < tries - 1:
                time.sleep(delay * (i + 1))
                continue
            return resp
        except Exception:
            if i == tries - 1:
                raise
            time.sleep(delay * (i + 1))


async def _aretry(fn, tries=5, delay=1.0):
    for i in range(tries):
        try:
            resp = await fn()
            if getattr(resp, 'status_code', 200) in _RETRY_STATUS and i < tries - 1:
                await asyncio.sleep(delay * (i + 1))
                continue
            return resp
        except Exception:
            if i == tries - 1:
                raise
            await asyncio.sleep(delay * (i + 1))

In [ ]:
# Example 1: public-API surface and instrumentor lifecycle (instrument/uninstrument idempotency).
from opentelemetry.instrumentation.httpx import HTTPXClientInstrumentor

instrumentor = HTTPXClientInstrumentor()
print(f"instrumentor type: {type(instrumentor).__name__}")
print(f"dependencies: {instrumentor.instrumentation_dependencies()}")

# instrument() and uninstrument() should be safely idempotent.
instrumentor.instrument()
assert instrumentor.is_instrumented_by_opentelemetry is True
instrumentor.uninstrument()
assert instrumentor.is_instrumented_by_opentelemetry is False

# Calling uninstrument again should not blow up.
instrumentor.uninstrument()
print("instrument/uninstrument lifecycle OK")

In [ ]:
# Example 2: tracing a real sync httpx.Client request and capturing the produced span.
# The tracer_provider is passed to instrument() explicitly — trace.set_tracer_provider()
# only honors its FIRST call per process, so the global provider is order-fragile.
import httpx
from opentelemetry.instrumentation.httpx import HTTPXClientInstrumentor
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))

HTTPXClientInstrumentor().instrument(tracer_provider=provider)
try:
    with httpx.Client(timeout=10.0) as client:
        resp = _retry(lambda: client.get('https://httpbingo.org/get'))
    spans = exporter.get_finished_spans()
    print(f"status: {resp.status_code}, spans captured: {len(spans)}")
    assert len(spans) >= 1
    print(f"span name: {spans[0].name}")
finally:
    HTTPXClientInstrumentor().uninstrument()

In [ ]:
# Example 3: per-client wrapping via instrument_client (does not patch globally).
import httpx
from opentelemetry.instrumentation.httpx import HTTPXClientInstrumentor

client = httpx.Client(timeout=10.0)
# Wrap just this specific client; global httpx remains untouched.
HTTPXClientInstrumentor.instrument_client(client)

try:
    resp = _retry(lambda: client.get('https://httpbingo.org/status/200'))
    print(f"per-client wrapped request -> {resp.status_code}")
    assert resp.status_code == 200
finally:
    HTTPXClientInstrumentor.uninstrument_client(client)
    client.close()

print("per-client instrument/uninstrument OK")

In [ ]:
# Example 4: request_hook + response_hook are invoked with the right argument shapes.
import httpx
from opentelemetry.instrumentation.httpx import HTTPXClientInstrumentor

seen = {"request": 0, "response": 0}

def request_hook(span, request_info):
    seen["request"] += 1

def response_hook(span, request_info, response_info):
    seen["response"] += 1

client = httpx.Client(timeout=10.0)
HTTPXClientInstrumentor.instrument_client(
    client, request_hook=request_hook, response_hook=response_hook,
)
try:
    _retry(lambda: client.get('https://httpbingo.org/status/204'))
finally:
    HTTPXClientInstrumentor.uninstrument_client(client)
    client.close()

print(f"hook counts: {seen}")
assert seen["request"] >= 1 and seen["response"] >= 1

In [ ]:
# Example 5: async client wrapping via instrument_client on an httpx.AsyncClient.
# tracer_provider goes to instrument_client directly — the global provider was already
# claimed by an earlier cell, so a second set_tracer_provider() would be a silent no-op.
import httpx
from opentelemetry.instrumentation.httpx import HTTPXClientInstrumentor
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))

async_client = httpx.AsyncClient(timeout=10.0)
HTTPXClientInstrumentor.instrument_client(async_client, tracer_provider=provider)
try:
    resp = await _aretry(lambda: async_client.get('https://httpbingo.org/get'))
    print(f"async status: {resp.status_code}")
    spans = exporter.get_finished_spans()
    print(f"async spans captured: {len(spans)}")
    assert resp.status_code == 200
    assert len(spans) >= 1
finally:
    HTTPXClientInstrumentor.uninstrument_client(async_client)
    await async_client.aclose()